# Miya MRT: o‘sma turini transformer bilan tasniflash

**Swin-T (torchvision, ImageNet) → figshare “brain tumor dataset” (Jun Cheng, CC BY 4.0) → MedNexus**

Bu notebook to‘rtta texnik savolga javob beradi va har bir javobni kod bilan ko‘rsatadi:

| Savol | Qisqa javob | Bo‘lim |
|---|---|---|
| Qaysi ochiq dataset? | figshare “brain tumor dataset” (Jun Cheng): 3 064 ta T1 kontrastli (T1-CE) MRT slice, 233 bemor, 3 sinf; CC BY 4.0, login shart emas | 1–2 |
| Qaysi transformer? | Swin Transformer Tiny (`torchvision.models.swin_t`, ImageNet-1K og‘irliklari); muqobil: ViT-B/16 | 4 |
| Qanday o‘qitiladi? | bemor darajasida bo‘lingan train/val/test, AdamW + warmup + cosine, label smoothing, mixed precision, class weights, val macro-F1 bo‘yicha early stopping | 3, 5 |
| Arxitektura? | patch embedding → 4 bosqichli (hierarchical) shifted-window self-attention → global pooling → Linear(768 → 3) | 4 |

**Raqamlar haqida.** Bu notebookda oldindan yozib qo‘yilgan birorta ham natija raqami yo‘q. Barcha metrikalar (accuracy, macro-F1, ROC-AUC, ECE, …) notebook ishga tushirilganda test bemorlarida hisoblanadi. Adabiyotdagi raqamlar faqat manbasi bilan keltirilgan (11-bo‘lim).

**Colab’da ishga tushirish:** *Runtime → Change runtime type → T4 GPU*, keyin *Runtime → Run all*. Qo‘shimcha paket kerak emas (torch, torchvision, h5py, scipy, scikit-learn, matplotlib, pandas Colab’da bor); faqat eksport bo‘limi `onnx` / `onnxruntime` topilmasa ularni `pip` bilan o‘rnatadi. Maqsad: T4’da taxminan 20–30 daqiqa (bu taxmin; yuklab olish, keshlash va har bir epoch vaqti chop etiladi).

**Mundarija:** 0 Sozlamalar · 1 Dataset · 2 `.mat` o‘qish va preprocessing · 3 Bemor darajasida bo‘lish · 4 Arxitektura · 5 O‘qitish · 6 Test metrikalari · 7 Kalibrlash · 8 Explainability · 9 Eksport va MedNexus · 10 Cheklovlar · 11 Manbalar

## 0. Sozlamalar (config)

Barcha sozlamalar shu katakda. `SMOKE_TEST = True` — kodni tez tekshirish rejimi: figshare’ning faqat 1-zip qismi, ~96 slice, 1 epoch; CPU’da ham ishlaydi, lekin natijalari **ma’nosiz**. `MODEL_NAME = "vit_b_16"` — ViT-B/16 bilan solishtirish uchun.

In [ ]:
# ============================ 0. Sozlamalar ============================
from pathlib import Path

SMOKE_TEST = False             # True: 1 ta zip qism, ~96 slice, 1 epoch (faqat kodni tekshirish)
MODEL_NAME = "swin_t"          # "swin_t" (asosiy) yoki "vit_b_16"
PRETRAINED = True              # ImageNet-1K og‘irliklari (transfer learning)
IMG_SIZE = 224                 # ikkala model ImageNet’da 224×224 da o‘qitilgan
EPOCHS = 40 if MODEL_NAME == "swin_t" else 20   # ViT-B/16 taxminan 3 barobar sekinroq
BATCH_SIZE = 32
LR = 1e-4 if MODEL_NAME == "swin_t" else 5e-5
WEIGHT_DECAY = 0.05
WARMUP_EPOCHS = 2
LABEL_SMOOTHING = 0.1
PATIENCE = 8                   # val macro-F1 shuncha epoch yaxshilanmasa — early stopping
GRAD_CLIP = 1.0
FREEZE_STAGES = 0              # 0: butun tarmoq fine-tune (GPU). CPU’da tezlik uchun 2: patch embedding + 1–2-bosqichlar muzlatiladi

SPLIT_MODE = "patient"         # "patient": 70/15/15 bemor bo‘yicha | "cvind": mualliflarning 5-fold indekslari
SPLIT_FRACTIONS = (0.70, 0.15, 0.15)
CV_TEST_FOLD = 1               # faqat "cvind": shu fold — test, keyingisi — val, qolgan 3 tasi — train
SEED = 42

# Augmentation (faqat train’da)
ROTATION_DEG = 10              # ±10°
HFLIP_P = 0.5                  # chap-o‘ng aks; vertikal aks ishlatilmaydi
CROP_SCALE = (0.85, 1.0)       # random resized crop: maydonning 85–100 %
CROP_RATIO = (0.9, 1.1)
GAMMA_RANGE = (0.8, 1.25)
CONTRAST_RANGE = (0.9, 1.1)
BRIGHTNESS_SHIFT = 0.05

# Preprocessing — MedNexus’da ham aynan shunday bo‘lishi shart (9-bo‘lim)
CLIP_PERCENTILES = (0.5, 99.5)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

DATA_DIR = Path("data/figshare_brain_tumor")
OUT_DIR = Path("outputs")
ZIP_PARTS = 4                  # figshare dataset 4 ta zip qismda
MAX_SLICES = None              # None — hammasi
CAM_EVAL_MAX = 400             # Grad-CAM + maska statistikasi uchun test slice’lar soni

if SMOKE_TEST:
    ZIP_PARTS, MAX_SLICES = 1, 96
    EPOCHS, BATCH_SIZE, WARMUP_EPOCHS, PATIENCE = 1, 8, 0, 1
    CAM_EVAL_MAX = 32
    OUT_DIR = Path("outputs_smoke")

assert MODEL_NAME in {"swin_t", "vit_b_16"}
assert MODEL_NAME != "vit_b_16" or IMG_SIZE == 224, "torchvision ViT-B/16 224 px uchun qurilgan"

In [ ]:
import copy, hashlib, importlib.util, inspect, json, math, random, shutil, subprocess, sys, time, urllib.request, warnings, zipfile
from collections import Counter

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.models as tvm
from PIL import Image
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, f1_score, roc_auc_score, roc_curve


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"            # mixed precision faqat GPU’da
torch.backends.cudnn.benchmark = True
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Sinf ranglari barcha grafiklarda bir xil
CLASS_COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.25, "grid.linewidth": 0.6, "figure.dpi": 100})

print(f"python {sys.version.split()[0]} | torch {torch.__version__} | torchvision {torchvision.__version__} | h5py {h5py.__version__}")
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
if DEVICE.type != "cuda" and not SMOKE_TEST:
    print("DIQQAT: GPU topilmadi. Colab: Runtime → Change runtime type → T4 GPU. CPU’da to‘liq o‘qitish soatlab davom etadi.")

## 1. Dataset: figshare “brain tumor dataset” (Jun Cheng)

- **Manba:** <https://figshare.com/articles/dataset/brain_tumor_dataset/1512427> · DOI [10.6084/m9.figshare.1512427](https://doi.org/10.6084/m9.figshare.1512427) (v8) · muallif: Jun Cheng. Login shart emas.
- **Litsenziya:** CC BY 4.0 (figshare sahifasi va API’da ko‘rsatilgan; quyidagi kod uni API’dan jonli o‘qiydi). Foydalanganda manbaga havola berish shart.
- **Tarkib (dataset README’si bo‘yicha):** 3 064 ta T1-weighted contrast-enhanced (Gd-DTPA) MRT slice, 233 bemor; meningioma — 708, glioma — 1 426, gipofiz (pituitary) o‘smasi — 930 slice. Axial, coronal va sagittal kesimlar. 512 × 512 piksel (0,49 × 0,49 mm), slice qalinligi 6 mm, oraliq 1 mm. Ikki shifoxona: Nanfang Hospital (Guangzhou) va General Hospital, Tianjin Medical University (Xitoy), 2005.9–2010.10.
- **Format:** har bir slice — alohida `.mat` fayl (MATLAB v7.3, ya’ni HDF5), `cjdata` strukturasi: `label` (1 = meningioma, 2 = glioma, 3 = pituitary), `PID` (bemor ID), `image`, `tumorBorder`, `tumorMask` (qo‘lda chizilgan o‘sma maskasi).
- **Fayllar:** 4 ta zip (har birida 766 slice, 214–232 MB, jami ≈ 880 MB), `cvind.mat` (mualliflarning 5-fold cross-validation indekslari), README. Kod har bir faylning md5 summasini figshare API’dagi qiymat bilan tekshiradi.

**Muqobil datasetlar (bu notebookda ishlatilmaydi):**
- **Kaggle “Brain Tumor MRI Dataset”** (Masoud Nickparvar, CC BY 4.0): 4 sinf, jumladan *no tumor*; hozirgi v2 versiyasida 7 200 rasm. Kaggle login kerak. U figshare, SARTAJ va Br35H to‘plamlaridan yig‘ilgan (*no tumor* rasmlari — Br35H’dan) va bemor ID’lari berilmagan, shuning uchun bemor darajasida bo‘lib bo‘lmaydi. Ichida figshare rasmlari ham bor: figshare’da o‘qitilgan modelni shu to‘plamda “tashqi test” qilish noto‘g‘ri.
- **BraTS** (Brain Tumor Segmentation challenge): 3D, to‘rtta ketma-ketlik (T1, T1-CE, T2, FLAIR), skull-stripping qilingan va umumiy anatomik shablonga registration qilingan; Synapse’da ro‘yxatdan o‘tish kerak. An’anaviy vazifa — glioma segmentatsiyasi (keyingi yillarda boshqa o‘sma turlari uchun ham challenge’lar qo‘shilgan). 3D va ko‘p ketma-ketlikli modelga o‘tish — keyingi qadam (10-bo‘lim).

In [ ]:
FIGSHARE_ARTICLE = 1512427
# nom: (figshare file id, md5) — api.figshare.com/v2/articles/1512427 (v8) dan
FIGSHARE_FILES = {
    "brainTumorDataPublic_1-766.zip": (3381290, "74b949ad33f042e6e103523091cd1428"),
    "brainTumorDataPublic_767-1532.zip": (3381296, "7e8a875500d2c8a346f270538e29890e"),
    "brainTumorDataPublic_1533-2298.zip": (3381293, "8227bf6080cb71f15a88be8d25c79ae7"),
    "brainTumorDataPublic_2299-3064.zip": (3381302, "b378a80d6174e5317d59eb28430c6652"),
    "cvind.mat": (7005344, "5ea82eb3212c9fcb84290ccf0beb486d"),
}
ZIP_NAMES = [name for name in FIGSHARE_FILES if name.endswith(".zip")][:ZIP_PARTS]


def md5sum(path, chunk=1 << 20):
    digest = hashlib.md5()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()


def download(name):
    file_id, md5 = FIGSHARE_FILES[name]
    target = DATA_DIR / name
    if target.exists() and md5sum(target) == md5:
        print(f"{name}: allaqachon bor, md5 to‘g‘ri")
        return target
    url = f"https://ndownloader.figshare.com/files/{file_id}"
    partial = target.with_name(target.name + ".part")
    start = time.time()
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (brain-mri-notebook)"})
    with urllib.request.urlopen(request, timeout=120) as response, open(partial, "wb") as out:
        shutil.copyfileobj(response, out, length=1 << 20)
    if md5sum(partial) != md5:
        partial.unlink()
        raise RuntimeError(f"{name}: md5 mos kelmadi — qayta yuklab ko‘ring")
    partial.replace(target)
    print(f"{name}: {target.stat().st_size / 1e6:.0f} MB, {time.time() - start:.0f} s, md5 to‘g‘ri")
    return target


# Nomi, litsenziyasi va md5’larni figshare API’dan jonli tekshiramiz
try:
    with urllib.request.urlopen(f"https://api.figshare.com/v2/articles/{FIGSHARE_ARTICLE}", timeout=30) as response:
        article = json.load(response)
except OSError as error:
    article = None
    print("figshare API’ga ulanib bo‘lmadi, yuqoridagi md5’lar ishlatiladi:", error)
if article:
    print(f"{article['title']} | litsenziya: {article['license']['name']} | doi:{article['doi']} | "
          f"muallif: {', '.join(a['full_name'] for a in article['authors'])}")
    api_md5 = {f["name"]: f["computed_md5"] for f in article["files"]}
    changed = [name for name, (_, md5) in FIGSHARE_FILES.items() if api_md5.get(name) != md5]
    print("md5’lar figshare API bilan mos" if not changed else f"DIQQAT: figshare’da o‘zgargan fayllar: {changed}")

for name in ZIP_NAMES + ["cvind.mat"]:
    download(name)

MAT_DIR = DATA_DIR / "mat"
MAT_DIR.mkdir(exist_ok=True)
for name in ZIP_NAMES:
    with zipfile.ZipFile(DATA_DIR / name) as archive:
        members = [m for m in archive.namelist() if m.endswith(".mat")]
        if not all((MAT_DIR / Path(m).name).exists() for m in members):
            for member in members:
                (MAT_DIR / Path(member).name).write_bytes(archive.read(member))
wanted = set()
for name in ZIP_NAMES:                                   # "brainTumorDataPublic_767-1532.zip" → 767..1532
    first_id, last_id = map(int, name.split("_")[1].removesuffix(".zip").split("-"))
    wanted.update(range(first_id, last_id + 1))
mat_files = sorted((p for p in MAT_DIR.glob("*.mat") if int(p.stem) in wanted), key=lambda p: int(p.stem))
MAT_BY_ID = {int(p.stem): p for p in mat_files}
print(len(mat_files), "ta .mat fayl")

## 2. `.mat` fayllarni o‘qish (MATLAB v7.3 = HDF5) va preprocessing

Fayl sarlavhasi `MATLAB 7.3 MAT-file … HDF5 schema` — bu HDF5 konteyner; `scipy.io.loadmat` uni o‘qimaydi (quyida ko‘rsatiladi), shuning uchun `h5py`. MATLAB massivlarni column-major saqlaydi va h5py ularni transponirlangan holda qaytaradi, shuning uchun `image.T` va `tumorMask.T` (tekshirilgan: transpozadan keyin kesimlar standart yo‘nalishda va maska o‘smaga tushadi).

**Preprocessing (har bir slice alohida — per-image):**
1. slice o‘zining 0,5 va 99,5 percentillari bo‘yicha [0, 1] ga keltiriladi — turli skaner va kontrast yorqinligi farqini kamaytiradi, yorqin outlier’larni kesadi;
2. kvadrat bo‘lmasa, qora fon bilan kvadratga to‘ldiriladi (figshare’da hammasi kvadrat);
3. bilinear resize → 224 × 224, uint8 sifatida keshlanadi;
4. modelga berishda grayscale 3 kanalga takrorlanadi va ImageNet mean/std bilan normallashtiriladi (pretrained og‘irliklar shu statistikani kutadi).

MedNexus’da ham aynan shu `preprocess_slice` funksiyasi ishlatilishi kerak (9-bo‘lim).

In [ ]:
CLASSES = ["meningioma", "glioma", "pituitary"]      # figshare label 1, 2, 3 → indeks 0, 1, 2


def read_meta(path):
    """(label 1..3, PID) — rasmni o‘qimasdan."""
    with h5py.File(path, "r") as f:
        return int(np.array(f["cjdata/label"]).item()), "".join(map(chr, np.array(f["cjdata/PID"]).ravel()))


def read_mat(path):
    """(image float32 H×W, o‘sma maskasi uint8 H×W, label 1..3, PID)."""
    with h5py.File(path, "r") as f:
        cj = f["cjdata"]
        image = np.array(cj["image"], dtype=np.float32).T          # MATLAB column-major → transpoze
        mask = np.array(cj["tumorMask"], dtype=np.uint8).T
        label = int(np.array(cj["label"]).item())
        pid = "".join(map(chr, np.array(cj["PID"]).ravel()))      # MATLAB char → uint16 kodlar
    return image, mask, label, pid


first = mat_files[0]
with open(first, "rb") as handle:
    print(handle.read(116).decode("latin-1").rstrip())
try:
    import scipy.io
    scipy.io.loadmat(first)
    print("scipy.io.loadmat o‘qidi")
except NotImplementedError as error:
    print("scipy.io.loadmat →", error)
with h5py.File(first, "r") as f:
    f["cjdata"].visititems(lambda name, obj: print(f"  cjdata/{name}: {obj.shape} {obj.dtype}"))
image, mask, label, pid = read_mat(first)
print(f"{first.name}: label={label} ({CLASSES[label - 1]}), PID={pid}, image {image.shape} "
      f"[{image.min():.0f}, {image.max():.0f}], maska ulushi {mask.mean():.3f}")

In [ ]:
def pad_to_square(a):
    h, w = a.shape
    s = max(h, w)
    out = np.zeros((s, s), dtype=a.dtype)
    out[(s - h) // 2:(s - h) // 2 + h, (s - w) // 2:(s - w) // 2 + w] = a
    return out


def preprocess_slice(image, size=IMG_SIZE):
    """Bitta slice → uint8 size×size: percentile min-max → kvadrat → bilinear resize."""
    img = np.asarray(image, dtype=np.float32)
    lo, hi = np.percentile(img, CLIP_PERCENTILES)
    img = np.clip((img - lo) / max(hi - lo, 1e-6), 0.0, 1.0)
    img = pad_to_square(img)
    u8 = Image.fromarray(np.round(img * 255).astype(np.uint8))
    return np.array(u8.resize((size, size), Image.Resampling.BILINEAR))


def preprocess_mask(mask, size=IMG_SIZE):
    m = Image.fromarray(pad_to_square(np.asarray(mask, dtype=np.uint8)) * 255)
    return (np.asarray(m.resize((size, size), Image.Resampling.NEAREST)) > 127).astype(np.uint8)


# 1) metadata (label, PID) — tez
start = time.time()
slice_ids_all = np.array([int(p.stem) for p in mat_files])
meta = [read_meta(p) for p in mat_files]
labels_all = np.array([m[0] for m in meta])
pids_all = np.array([m[1] for m in meta])
mixed = [p for p in np.unique(pids_all) if len(set(labels_all[pids_all == p])) > 1]
print(f"metadata: {len(mat_files)} slice, {len(np.unique(pids_all))} bemor, {time.time() - start:.0f} s; "
      f"bir nechta o‘sma turi bor bemorlar: {mixed or 'yo‘q'}")


def subsample_patients(pids, labels, max_slices, seed):
    """Smoke test uchun: butun bemorlarni sinflar bo‘yicha navbatma-navbat olamiz."""
    rng = np.random.default_rng(seed)
    queues = {c: list(rng.permutation(np.unique(pids[labels == c]))) for c in np.unique(labels)}
    chosen, n = set(), 0
    while n < max_slices and any(queues.values()):
        for c in queues:
            if queues[c] and n < max_slices:
                p = queues[c].pop()
                if p not in chosen:
                    chosen.add(p)
                    n += int((pids == p).sum())
    return np.isin(pids, list(chosen))


keep = subsample_patients(pids_all, labels_all, MAX_SLICES, SEED) if MAX_SLICES else np.ones(len(mat_files), bool)
sel = np.flatnonzero(keep)

# 2) rasm + maska → preprocessing → uint8 kesh (.npz)
cache_path = DATA_DIR / f"cache_{IMG_SIZE}px_{len(sel)}slices.npz"
if cache_path.exists():
    cache = np.load(cache_path)
    assert np.array_equal(cache["slice_ids"], slice_ids_all[sel]), "kesh boshqa slice’lar uchun — uni o‘chiring"
    images, masks = cache["images"], cache["masks"]
    print("kesh o‘qildi:", cache_path)
else:
    images = np.zeros((len(sel), IMG_SIZE, IMG_SIZE), np.uint8)
    masks = np.zeros_like(images)
    shapes, start = Counter(), time.time()
    for k, i in enumerate(sel):
        image, mask, _, _ = read_mat(mat_files[i])
        shapes[image.shape] += 1
        images[k], masks[k] = preprocess_slice(image), preprocess_mask(mask)
        if (k + 1) % 500 == 0:
            print(f"  {k + 1}/{len(sel)} ({time.time() - start:.0f} s)")
    print(f"asl o‘lchamlar: {dict(shapes)} | {time.time() - start:.0f} s")
    np.savez(cache_path, images=images, masks=masks, slice_ids=slice_ids_all[sel])

y = labels_all[sel] - 1                 # 0 = meningioma, 1 = glioma, 2 = pituitary
pids = pids_all[sel]
slice_ids = slice_ids_all[sel]
print("images:", images.shape, images.dtype, "| masks:", masks.shape)

In [ ]:
counts = pd.DataFrame({
    "slice": np.bincount(y, minlength=len(CLASSES)),
    "bemor": [len(np.unique(pids[y == c])) for c in range(len(CLASSES))],
}, index=CLASSES)
counts.loc["jami"] = [len(y), len(np.unique(pids))]
display(counts)
per_patient = pd.Series(pids).value_counts()
print(f"slice / bemor: min {per_patient.min()}, median {per_patient.median():.0f}, max {per_patient.max()}")
if not SMOKE_TEST:
    expected = {"meningioma": 708, "glioma": 1426, "pituitary": 930}
    matches = all(counts.loc[c, "slice"] == n for c, n in expected.items()) and counts.loc["jami", "bemor"] == 233
    print("README bilan mos: 3064 slice, 233 bemor, 708 / 1426 / 930" if matches else "DIQQAT: sonlar README’dan farq qiladi")

rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(len(CLASSES), 4, figsize=(12, 9))
for c, name in enumerate(CLASSES):
    options = np.flatnonzero(y == c)
    picks = rng.choice(options, size=min(4, len(options)), replace=False) if len(options) else []
    for j, ax in enumerate(axes[c]):
        ax.axis("off")
        if j < len(picks):
            i = picks[j]
            ax.imshow(images[i], cmap="gray")
            ax.contour(masks[i], levels=[0.5], colors=[CLASS_COLORS[c]], linewidths=1.2)
            ax.set_title(f"{name} · PID {pids[i]}", fontsize=9)
fig.suptitle("Har bir sinfdan misollar (kontur — dataset’dagi o‘sma maskasi)")
plt.tight_layout()
plt.show()

## 3. Bemor darajasida bo‘lish (patient-level split)

**Nega slice darajasidagi tasodifiy bo‘lish noto‘g‘ri.** Bitta bemordan o‘rtacha ~13 slice bor (3 064 / 233). Bir bemorning slice’lari bir xil anatomiya, bir xil skaner va kontrast, bir xil o‘smani ko‘rsatadi, qo‘shni slice’lar esa deyarli nusxa. Tasodifiy slice split’da bemorning bir slice’i train’ga, qo‘shnisi test’ga tushadi va model o‘sma turini emas, *aynan shu bemorni* tanib olishi mumkin. Natijada test aniqligi yangi bemorlardagi haqiqiy aniqlikdan yuqori chiqadi (data leakage). Badža & Barjaktarović (2020) aynan shu datasetda 10-fold record-wise (slice bo‘yicha) va subject-wise (bemor bo‘yicha) cross-validation’ni solishtirgan: ularning eng yuqori natijasi (96,56 %) record-wise + augmentation holatida olingan, subject-wise natijalar undan past.

**Bizning yondashuv.** Bemorlar (PID) train/val/test’ga 70/15/15 nisbatda, har bir sinf ichida alohida (stratified) bo‘linadi — har bir splitda uchala o‘sma turi bo‘ladi. Bitta bemorning barcha slice’lari faqat bitta splitda. Bemorlardagi slice soni har xil bo‘lgani uchun slice ulushlari 70/15/15 dan biroz farq qiladi.
Muqobil: `SPLIT_MODE = "cvind"` — mualliflarning `cvind.mat` faylidagi 5-fold indekslari (ular ham bemor darajasida — kod buni tekshiradi); adabiyot bilan solishtirish uchun qulay.

Test bemorlari faqat oxirida bir marta ishlatiladi: early stopping va temperature scaling faqat val’da tanlanadi.

In [ ]:
def patient_split(pids, y, fractions, seed):
    """Har bir sinf ichida bemorlarni aralashtirib, train/val/test’ga ajratamiz."""
    rng = np.random.default_rng(seed)
    patients = np.unique(pids)
    patient_label = {p: np.bincount(y[pids == p]).argmax() for p in patients}   # figshare’da bemor = bitta o‘sma turi
    assign = {}
    for c in sorted(set(patient_label.values())):
        group = rng.permutation([p for p in patients if patient_label[p] == c])
        n = len(group)
        n_test, n_val = int(round(n * fractions[2])), int(round(n * fractions[1]))
        if n >= 3:
            n_test, n_val = max(n_test, 1), max(n_val, 1)
        for k, p in enumerate(group):
            assign[p] = "test" if k < n_test else "val" if k < n_test + n_val else "train"
    return np.array([assign[p] for p in pids])


def cvind_split(slice_ids, test_fold):
    with h5py.File(DATA_DIR / "cvind.mat", "r") as f:
        fold = np.array(f["cvind"]).ravel().astype(int)[slice_ids - 1]
    val_fold = test_fold % 5 + 1
    return np.where(fold == test_fold, "test", np.where(fold == val_fold, "val", "train"))


with h5py.File(DATA_DIR / "cvind.mat", "r") as f:
    cv_fold = np.array(f["cvind"]).ravel().astype(int)[slice_ids - 1]
spanning = sum(len(set(cv_fold[pids == p])) > 1 for p in np.unique(pids))
print(f"cvind.mat: {spanning} ta bemor bir nechta fold’ga tushgan (0 bo‘lsa — indekslar bemor darajasida)")

split = patient_split(pids, y, SPLIT_FRACTIONS, SEED) if SPLIT_MODE == "patient" else cvind_split(slice_ids, CV_TEST_FOLD)
idx_train, idx_val, idx_test = (np.flatnonzero(split == s) for s in ("train", "val", "test"))
sets = {s: set(pids[split == s]) for s in ("train", "val", "test")}
assert not (sets["train"] & sets["val"] or sets["train"] & sets["test"] or sets["val"] & sets["test"]), "bemor ikki splitda!"
assert len(idx_val) and len(idx_test), "val yoki test bo‘sh"

table = pd.DataFrame({s: [int((y[split == s] == c).sum()) for c in range(len(CLASSES))] + [int((split == s).sum()), len(sets[s])]
                      for s in ("train", "val", "test")}, index=[f"{c} (slice)" for c in CLASSES] + ["jami slice", "bemor"])
display(table)
print("Bemorlar kesishmasi: yo‘q (assert o‘tdi)")

# Taqqoslash: agar slice’larni tasodifiy bo‘lganimizda
perm = np.random.default_rng(SEED).permutation(len(y))
n_train, n_val_r = int(SPLIT_FRACTIONS[0] * len(y)), int(SPLIT_FRACTIONS[1] * len(y))
leaked = np.isin(pids[perm[n_train + n_val_r:]], pids[perm[:n_train]]).mean()
print(f"Taqqoslash: tasodifiy slice split’da test slice’larining {leaked:.0%} qismi train’da ham slice’i bor bemorlarga "
      "tegishli bo‘lardi (bizning split’da — 0 %).")

## 4. Model arxitekturasi: Swin Transformer (Tiny)

Swin Transformer (Liu va boshq., ICCV 2021) — ierarxik vision transformer. Asosiy g‘oyalar:

1. **Patch embedding.** 224 × 224 × 3 tasvir 4 × 4 piksellik patch’larga bo‘linadi (56 × 56 = 3 136 token), har biri 96 o‘lchamli vektorga proyeksiya qilinadi (amalda: 4 × 4, stride 4 li Conv2d + LayerNorm).
2. **Window self-attention (W-MSA).** Self-attention butun tasvir bo‘ylab emas, 7 × 7 tokenli oynalar ichida hisoblanadi — hisoblash hajmi token soniga chiziqli bog‘liq (global attention’da kvadratik). Attention’ga nisbiy pozitsiya bias’i qo‘shiladi: softmax(QKᵀ/√d + B)·V.
3. **Shifted windows (SW-MSA).** Har ikkinchi blokda oynalar 3 tokenga (oynaning yarmi) siljitiladi — qo‘shni oynalar ma’lumot almashadi, receptive field kengayadi.
4. **Ierarxik bosqichlar.** Har bosqichdan keyin *patch merging* 2 × 2 qo‘shni tokenni birlashtiradi: o‘lcham 56 → 28 → 14 → 7, kanallar 96 → 192 → 384 → 768 (CNN piramidasi kabi). Mayda tekstura ham, katta kontekst ham (o‘smaning joyi: sella, dura, parenxima) ko‘riladi.
5. **Classification head.** LayerNorm → global average pooling → Linear(768 → 3). ImageNet’ning 1000 sinfli head’i olib tashlanib, yangi 3 sinfli head qo‘yiladi.

```
 Kirish: 224×224 MRT slice (grayscale → 3 kanal, ImageNet normalizatsiya)
   │ patch partition 4×4 + linear embedding (Conv 4×4, stride 4 → 96 kanal)
   ▼
 56×56×96   ─ Stage 1: 2 × Swin block,  3 head      Swin block’lar juft-juft:
   │ patch merging (2×2 → 1, kanal ×2)                 x → LN → W-MSA  → (+) → LN → MLP → (+)
 28×28×192  ─ Stage 2: 2 × Swin block,  6 head         x → LN → SW-MSA → (+) → LN → MLP → (+)
   │ patch merging                                     W-MSA: 7×7 oyna ichida attention
 14×14×384  ─ Stage 3: 6 × Swin block, 12 head         SW-MSA: 3 tokenga siljigan oynalar
   │ patch merging
  7×7×768   ─ Stage 4: 2 × Swin block, 24 head   ← Grad-CAM shu bosqichga ulanadi (8-bo‘lim)
   │ LayerNorm → global average pool → Linear(768 → 3)
   ▼
 logits [meningioma, glioma, pituitary] → softmax(logits / T)   (T — temperature, 7-bo‘lim)
```

**Nega transfer learning.** Train’da taxminan 2 100 slice (~160 bemor) bor. Transformer’larda CNN’dagi kabi locality va translation-equivariance inductive bias’i yo‘q, noldan o‘qitilganda ular juda ko‘p ma’lumot talab qiladi (ViT maqolasi, Dosovitskiy va boshq., 2021). ImageNet-1K’da oldindan o‘qitilgan og‘irliklar chegara, tekstura va shakl kabi umumiy vizual belgilarni allaqachon biladi; biz butun tarmoqni kichik learning rate bilan fine-tune qilamiz.

**Muqobil — ViT-B/16** (`MODEL_NAME = "vit_b_16"`): 16 × 16 patch → 14 × 14 = 196 token + [CLS], 12 ta global self-attention qatlami (12 head, 768 o‘lcham), head — [CLS] tokendan Linear. Parametrlari Swin-T’dan ~3 barobar ko‘p. MedNexus’dagi mavjud 4 sinfli model ham ViT-B/16 (Hugging Face `Hemg/Brain-Tumor-Classification`). Explainability — attention rollout.

Parametrlar soni va ImageNet top-1 quyida torchvision metama’lumotidan chop etiladi.

In [ ]:
def build_model(name, num_classes, pretrained=True):
    if name == "swin_t":
        weights = tvm.Swin_T_Weights.IMAGENET1K_V1 if pretrained else None
        model = tvm.swin_t(weights=weights)                      # stochastic depth 0.2 (torchvision standarti)
        model.head = nn.Linear(model.head.in_features, num_classes)
    else:
        weights = tvm.ViT_B_16_Weights.IMAGENET1K_V1 if pretrained else None
        model = tvm.vit_b_16(weights=weights)
        model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    return model, weights


model, weights = build_model(MODEL_NAME, len(CLASSES), PRETRAINED)
model = model.to(DEVICE).eval()
if FREEZE_STAGES:      # ixtiyoriy: pastki qatlamlar ImageNet og‘irliklarida qoladi, backward ularga yetmaydi (CPU’da tezroq)
    if MODEL_NAME == "swin_t":
        frozen = list(model.features[:2 * FREEZE_STAGES])        # [patch embedding, stage 1, patch merging, stage 2, ...]
    else:
        frozen = [model.conv_proj, *model.encoder.layers[:3 * FREEZE_STAGES]]
        model.class_token.requires_grad_(False)
        model.encoder.pos_embedding.requires_grad_(False)
    for module in frozen:
        module.requires_grad_(False)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{MODEL_NAME}: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M parametr (3 sinfli head bilan), "
      f"o‘qitiladigan: {trainable / 1e6:.1f} M" + (f" (FREEZE_STAGES = {FREEZE_STAGES})" if FREEZE_STAGES else ""))
if weights is not None:
    print(f"og‘irliklar: {weights} | ImageNet-1K top-1 {weights.meta['_metrics']['ImageNet-1K']['acc@1']} % "
          f"(torchvision metama’lumoti, 1000 sinfli head bilan {weights.meta['num_params'] / 1e6:.1f} M)")

with torch.no_grad():                                           # tensor o‘lchamlari bosqichma-bosqich
    x = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    if MODEL_NAME == "swin_t":
        names = ["patch embedding", "stage 1", "patch merging", "stage 2", "patch merging", "stage 3", "patch merging", "stage 4"]
        for name, layer in zip(names, model.features):
            x = layer(x)
            print(f"{name:<16} → {tuple(x.shape)}  (B, H, W, C)")
        print(f"{'norm+pool+head':<16} → {tuple(model(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)).shape)}")
    else:
        tokens = model._process_input(x)
        print(f"patch embedding → {tuple(tokens.shape)}  (B, 196 token, 768); + [CLS] → 197 token → 12 × encoder")
        print(f"[CLS] → head    → {tuple(model(x).shape)}")

## 5. O‘qitish (training)

| Element | Tanlov | Nega |
|---|---|---|
| Kirish | 224 px, grayscale → 3 kanal, per-image percentile normalizatsiya + ImageNet mean/std | pretrained og‘irliklar bilan moslik |
| Augmentation | ±10° burish; chap-o‘ng aks (p = 0,5); random resized crop (maydon 85–100 %, nisbat 0,9–1,1); gamma 0,8–1,25; kontrast ±10 %; yorqinlik ±0,05 | skanerlar va joylashuvdagi real farqlarga o‘xshash. Vertikal aks yo‘q (miya teskari turmaydi). Sagittal kesimda chap-o‘ng aks old va orqani almashtiradi — nostandart ko‘rinish, lekin sinfni o‘zgartirmaydi; xohlasangiz `HFLIP_P = 0` |
| Loss | cross-entropy + class weights (N / (K·nₖ), train’dan) + label smoothing 0,1 | glioma slice’lari meningiomadan ~2 barobar ko‘p |
| Optimizer | AdamW, lr 1e-4 (ViT: 5e-5), weight decay 0,05 (LayerNorm, bias va pozitsiya jadvallariga — 0) | transformer fine-tuning uchun odatiy tanlov |
| Schedule | chiziqli warmup (2 epoch) → cosine | boshida yangi head’ning katta gradientlari pretrained og‘irliklarni buzmasligi uchun |
| Aniqlik | mixed precision (fp16 autocast + GradScaler) GPU’da; gradient clipping 1,0 | tezroq va xotira kamroq |
| To‘xtatish | val macro-F1 bo‘yicha early stopping (patience 8); eng yaxshi checkpoint saqlanadi | sinflar nomutanosib — accuracy emas, macro-F1 |
| Muzlatish (ixtiyoriy) | `FREEZE_STAGES = 0` — butun tarmoq o‘qitiladi. GPU bo‘lmasa `2`: patch embedding va 1–2-bosqichlar ImageNet og‘irliklarida qoladi | CPU’da bir training step ~30 % tezroq (i7-1165G7 da o‘lchangan); pastki qatlamlardagi umumiy belgilar (chegara, tekstura) ImageNet’dan yaxshi ko‘chadi |

Augmentation GPU’da butun batch uchun bitta affine almashtirish (`affine_grid` + `grid_sample`) bilan qilinadi, shuning uchun Colab’ning 2 ta CPU yadrosi bottleneck bo‘lmaydi. Keshlangan dataset (uint8, ~150 MB) to‘liq GPU xotirasida turadi.

In [ ]:
MEAN = torch.tensor(IMAGENET_MEAN, device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor(IMAGENET_STD, device=DEVICE).view(1, 3, 1, 1)
X_ALL = torch.from_numpy(images).to(DEVICE)                   # uint8 (N, H, W)
Y_ALL = torch.from_numpy(y).long().to(DEVICE)
tidx_train, tidx_val, tidx_test = (torch.as_tensor(i, device=DEVICE) for i in (idx_train, idx_val, idx_test))


def to_model_input(x01):
    """(B, 1, H, W) [0, 1] → (B, 3, H, W): grayscale 3 kanalga takrorlanadi, ImageNet mean/std."""
    return (x01.expand(-1, 3, -1, -1) - MEAN.to(x01.device)) / STD.to(x01.device)


def augment(x):
    """Har bir rasm uchun alohida tasodifiy: burish + chap-o‘ng aks + resized crop (bitta affine), gamma/kontrast/yorqinlik."""
    b, dev = x.shape[0], x.device
    u = lambda lo, hi: torch.empty(b, device=dev).uniform_(lo, hi)
    angle = u(-ROTATION_DEG, ROTATION_DEG) * math.pi / 180
    area, log_ratio = u(*CROP_SCALE), u(math.log(CROP_RATIO[0]), math.log(CROP_RATIO[1]))
    sx = torch.sqrt(area * log_ratio.exp()).clamp(max=1.0)       # crop kengligi (butun rasmga nisbatan)
    sy = torch.sqrt(area / log_ratio.exp()).clamp(max=1.0)
    tx, ty = u(-1, 1) * (1 - sx), u(-1, 1) * (1 - sy)             # crop markazi rasm ichida
    flip = torch.where(torch.rand(b, device=dev) < HFLIP_P, -1.0, 1.0)
    cos, sin = torch.cos(angle), torch.sin(angle)
    theta = torch.stack([torch.stack([cos * sx * flip, -sin * sy, tx], 1),
                         torch.stack([sin * sx * flip, cos * sy, ty], 1)], 1)       # (B, 2, 3): R · S · Flip + t
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    x = x.clamp(0, 1) ** u(*GAMMA_RANGE).view(b, 1, 1, 1)
    x = x * u(*CONTRAST_RANGE).view(b, 1, 1, 1) + u(-BRIGHTNESS_SHIFT, BRIGHTNESS_SHIFT).view(b, 1, 1, 1)
    return x.clamp(0, 1)


def get_batch(idx, train):
    x = X_ALL[idx].unsqueeze(1).float().div_(255)
    if train:
        x = augment(x)
    return to_model_input(x), Y_ALL[idx]


demo = X_ALL[tidx_train[:1]].unsqueeze(1).float().div(255).repeat(8, 1, 1, 1)
fig, axes = plt.subplots(1, 9, figsize=(18, 2.4))
for ax, img, title in zip(axes, [demo[0]] + list(augment(demo)), ["asl"] + [f"aug {k}" for k in range(1, 9)]):
    ax.imshow(img[0].cpu(), cmap="gray", vmin=0, vmax=1)
    ax.set_title(title, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
train_counts = np.bincount(y[idx_train], minlength=len(CLASSES))
class_weights = torch.tensor(len(idx_train) / (len(CLASSES) * np.maximum(train_counts, 1)), dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
print("class weights:", {c: round(float(w), 3) for c, w in zip(CLASSES, class_weights)})

no_decay_keys = ("relative_position_bias_table", "pos_embedding", "class_token")
decay, no_decay = [], []
for name, param in model.named_parameters():
    if not param.requires_grad:                                  # FREEZE_STAGES > 0 bo‘lsa
        continue
    (no_decay if param.ndim <= 1 or any(k in name for k in no_decay_keys) else decay).append(param)
optimizer = torch.optim.AdamW([{"params": decay, "weight_decay": WEIGHT_DECAY},
                               {"params": no_decay, "weight_decay": 0.0}], lr=LR)

steps_per_epoch = math.ceil(len(idx_train) / BATCH_SIZE)
total_steps, warmup_steps = EPOCHS * steps_per_epoch, WARMUP_EPOCHS * steps_per_epoch


def lr_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)


@torch.no_grad()
def predict_logits(model, idx, batch_size=64):
    model.eval()
    out = []
    for i in range(0, len(idx), batch_size):
        x, _ = get_batch(idx[i:i + batch_size], train=False)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            out.append(model(x).float())
    return torch.cat(out).cpu()


def macro_f1(labels, pred):
    return f1_score(labels, pred, average="macro", labels=list(range(len(CLASSES))), zero_division=0)


ckpt_path = OUT_DIR / f"{MODEL_NAME}_best.pt"
history, best_f1, bad_epochs = [], -1.0, 0
generator = torch.Generator().manual_seed(SEED)
train_start = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_start, loss_sum = time.time(), 0.0
    order = tidx_train[torch.randperm(len(tidx_train), generator=generator).to(DEVICE)]
    for i in range(0, len(order), BATCH_SIZE):
        xb, yb = get_batch(order[i:i + BATCH_SIZE], train=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            loss = criterion(model(xb), yb)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        loss_sum += loss.item() * len(yb)

    val_logits = predict_logits(model, tidx_val)
    val_pred = val_logits.argmax(1).numpy()
    record = {"epoch": epoch, "train_loss": loss_sum / len(idx_train),
              "val_nll": F.cross_entropy(val_logits, torch.from_numpy(y[idx_val])).item(),
              "val_acc": float((val_pred == y[idx_val]).mean()), "val_macro_f1": macro_f1(y[idx_val], val_pred),
              "lr": optimizer.param_groups[0]["lr"], "seconds": time.time() - epoch_start}
    history.append(record)
    improved = record["val_macro_f1"] > best_f1
    if improved:
        best_f1, bad_epochs = record["val_macro_f1"], 0
        torch.save({"model": model.state_dict(), "epoch": epoch, "val_macro_f1": best_f1,
                    "model_name": MODEL_NAME, "classes": CLASSES, "img_size": IMG_SIZE}, ckpt_path)
    else:
        bad_epochs += 1
    print(f"epoch {epoch:02d} | train loss {record['train_loss']:.3f} | val NLL {record['val_nll']:.3f} | "
          f"val acc {record['val_acc']:.3f} | val macro-F1 {record['val_macro_f1']:.3f} | lr {record['lr']:.1e} | "
          f"{record['seconds']:.0f} s{'  ← eng yaxshi, saqlandi' if improved else ''}")
    if bad_epochs >= PATIENCE:
        print(f"early stopping: {PATIENCE} epoch davomida val macro-F1 yaxshilanmadi")
        break
print(f"o‘qitish: {(time.time() - train_start) / 60:.1f} daqiqa")

In [ ]:
hist = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(hist.epoch, hist.train_loss, lw=2, marker="o", ms=4, color=CLASS_COLORS[0], label="train loss (smoothing + weights)")
axes[0].plot(hist.epoch, hist.val_nll, lw=2, marker="o", ms=4, color=CLASS_COLORS[1], label="val NLL")
axes[0].set(xlabel="epoch", title="Loss")
axes[0].legend(frameon=False)
axes[1].plot(hist.epoch, hist.val_acc, lw=2, marker="o", ms=4, color=CLASS_COLORS[0], label="val accuracy")
axes[1].plot(hist.epoch, hist.val_macro_f1, lw=2, marker="o", ms=4, color=CLASS_COLORS[1], label="val macro-F1")
axes[1].set(xlabel="epoch", ylim=(0, 1.02), title="Validation")
axes[1].legend(frameon=False)
plt.tight_layout()
plt.show()

checkpoint = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
model.load_state_dict(checkpoint["model"])
model.eval()
print(f"eng yaxshi checkpoint: epoch {checkpoint['epoch']}, val macro-F1 {checkpoint['val_macro_f1']:.3f}")

## 6. Test bemorlarida baholash

Faqat held-out test bemorlarida: accuracy, macro-F1, balanced accuracy; har bir sinf uchun sensitivity (recall), specificity, PPV va one-vs-rest ROC-AUC; confusion matrix. Test to‘plami kichik (~35 bemor), shuning uchun bemorlar bo‘yicha bootstrap 95 % ishonch oralig‘i ham hisoblanadi (bemorlar qaytarib tanlanadi, slice’lar bemor bilan birga). Qo‘shimcha: bemor darajasidagi aniqlik (bemor slice’lari ehtimollarining o‘rtachasi bo‘yicha).

**Adabiyot bilan solishtirish haqida.** Cheng va boshq. (PLoS ONE, 2015) shu datasetda bemor darajasidagi 5-fold CV bilan 91,28 % accuracy olgan (BoW + kengaytirilgan o‘sma ROI + ring-form partition). Lekin ularning usuli test vaqtida ham qo‘lda chizilgan o‘sma chegarasini talab qiladi, bizning model esa butun slice’ni ko‘radi va bitta split’da baholanadi — raqamlar to‘g‘ridan-to‘g‘ri solishtirilmaydi. Solishtirish uchun `SPLIT_MODE = "cvind"` bilan 5 fold’ni ham ishga tushiring.

In [ ]:
test_logits = predict_logits(model, tidx_test)
val_logits = predict_logits(model, tidx_val)
y_test, y_val = y[idx_test], y[idx_val]
test_probs = torch.softmax(test_logits, 1).numpy()
test_pred = test_probs.argmax(1)
K = len(CLASSES)
cm = confusion_matrix(y_test, test_pred, labels=list(range(K)))

rows = []
for c, name in enumerate(CLASSES):
    tp = cm[c, c]
    fn, fp = cm[c].sum() - tp, cm[:, c].sum() - tp
    tn = cm.sum() - tp - fn - fp
    positive = (y_test == c).astype(int)
    rows.append({"sinf": name, "test slice": int(cm[c].sum()),
                 "sensitivity": tp / (tp + fn) if tp + fn else np.nan,
                 "specificity": tn / (tn + fp) if tn + fp else np.nan,
                 "PPV": tp / (tp + fp) if tp + fp else np.nan,
                 "ROC-AUC (OvR)": roc_auc_score(positive, test_probs[:, c]) if 0 < positive.sum() < len(positive) else np.nan})
per_class = pd.DataFrame(rows).set_index("sinf")
display(per_class.round(3))

# Bemor darajasi: har bir test bemorining slice ehtimollari o‘rtachasi
test_pids = pids[idx_test]
patient_ids = np.unique(test_pids)
patient_true = np.array([np.bincount(y_test[test_pids == p]).argmax() for p in patient_ids])
patient_pred = np.array([test_probs[test_pids == p].mean(0).argmax() for p in patient_ids])

# Bemorlar bo‘yicha bootstrap 95 % CI
rng = np.random.default_rng(SEED)
groups = {p: np.flatnonzero(test_pids == p) for p in patient_ids}
boot_acc, boot_f1 = [], []
for _ in range(1000):
    ii = np.concatenate([groups[p] for p in rng.choice(patient_ids, size=len(patient_ids), replace=True)])
    boot_acc.append((test_pred[ii] == y_test[ii]).mean())
    boot_f1.append(macro_f1(y_test[ii], test_pred[ii]))

results = {
    "test_slices": len(idx_test), "test_patients": len(patient_ids),
    "accuracy": float((test_pred == y_test).mean()), "accuracy_ci95": np.percentile(boot_acc, [2.5, 97.5]).round(3).tolist(),
    "macro_f1": float(macro_f1(y_test, test_pred)), "macro_f1_ci95": np.percentile(boot_f1, [2.5, 97.5]).round(3).tolist(),
    "balanced_accuracy": float(balanced_accuracy_score(y_test, test_pred)),
    "macro_roc_auc_ovr": float(np.nanmean(per_class["ROC-AUC (OvR)"])) if per_class["ROC-AUC (OvR)"].notna().any() else float("nan"),
    "patient_level_accuracy": float((patient_pred == patient_true).mean()),
}
for key, value in results.items():
    print(f"{key:>24}: {value:.3f}" if isinstance(value, float) else f"{key:>24}: {value}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
row_norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
for ax, matrix, title in ((axes[0], cm, "Confusion matrix (slice soni)"), (axes[1], row_norm, "Qator bo‘yicha ulush (= sensitivity diagonalda)")):
    ax.imshow(row_norm, cmap="Blues", vmin=0, vmax=1)
    for i in range(K):
        for j in range(K):
            text = f"{matrix[i, j]:.2f}" if matrix is row_norm else str(matrix[i, j])
            ax.text(j, i, text, ha="center", va="center", color="white" if row_norm[i, j] > 0.6 else "#0b0b0b", fontsize=11)
    ax.set(xticks=range(K), yticks=range(K), xticklabels=CLASSES, yticklabels=CLASSES,
           xlabel="bashorat", ylabel="haqiqiy", title=title)
    ax.grid(False)
for c, name in enumerate(CLASSES):
    positive = (y_test == c).astype(int)
    if 0 < positive.sum() < len(positive):
        fpr, tpr, _ = roc_curve(positive, test_probs[:, c])
        axes[2].plot(fpr, tpr, lw=2, color=CLASS_COLORS[c], label=f"{name} (AUC {per_class.loc[name, 'ROC-AUC (OvR)']:.3f})")
axes[2].plot([0, 1], [0, 1], "--", color="#999999", lw=1)
axes[2].set(xlabel="1 − specificity (FPR)", ylabel="sensitivity (TPR)", title="ROC, one-vs-rest (test)", aspect="equal")
if axes[2].get_legend_handles_labels()[0]:
    axes[2].legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

## 7. Kalibrlash: “model qanchalik ishonchli?”

Softmax chiqishi avtomatik ravishda ehtimollik emas: model “0,95” desa, shunday holatlarning taxminan 95 % ida to‘g‘ri bo‘lishi kerak — buni tekshirish kerak.

- **Reliability diagram:** test slice’lari ishonch (max softmax) bo‘yicha 15 ta oraliqqa bo‘linadi; har oraliqda haqiqiy aniqlik o‘rtacha ishonch bilan solishtiriladi. Ideal holda ustunlar diagonalda.
- **ECE** (expected calibration error) = Σ_b (n_b / N) · |aniqlik(b) − ishonch(b)|.
- **Temperature scaling** (Guo va boshq., ICML 2017): logit’lar bitta skalyar T ga bo‘linadi, T faqat **val**’da NLL’ni minimallashtirib topiladi. Argmax o‘zgarmaydi (accuracy va F1 ham), faqat ishonch darajasi to‘g‘rilanadi. T > 1 — model o‘ta ishonchli edi (over-confident), T < 1 — yetarlicha ishonchli emas edi. Label smoothing ishonchni pasaytiradi (Müller va boshq., NeurIPS 2019), shuning uchun T < 1 chiqishi ham mumkin — aynan shuning uchun kalibrlashni o‘lchaymiz.

MedNexus’da ko‘rsatiladigan ehtimollar yuqoridan cheklangan va “taxmin” deb belgilanadi; kalibrlash — bunga texnik asos. Test’da bir necha yuz slice bor, shuning uchun ECE’ning o‘zi ham shovqinli baho.

In [ ]:
def calibration_bins(probs, labels, n_bins=15):
    conf, correct = probs.max(1), probs.argmax(1) == labels
    edges, bins, ece = np.linspace(0, 1, n_bins + 1), [], 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            bins.append((lo, int(m.sum()), correct[m].mean(), conf[m].mean()))
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return ece, bins


def fit_temperature(logits, labels):
    log_t = torch.zeros(1, requires_grad=True)
    lbfgs = torch.optim.LBFGS([log_t], lr=0.1, max_iter=200, line_search_fn="strong_wolfe")
    target = torch.as_tensor(labels)

    def closure():
        lbfgs.zero_grad()
        loss = F.cross_entropy(logits / log_t.exp(), target)
        loss.backward()
        return loss

    lbfgs.step(closure)
    return float(log_t.detach().exp().clamp(0.05, 20.0))


def calibration_report(logits, labels, t):
    probs = torch.softmax(logits / t, 1).numpy()
    onehot = np.eye(K)[labels]
    return {"NLL": F.cross_entropy(logits / t, torch.as_tensor(labels)).item(),
            "ECE": calibration_bins(probs, labels)[0],
            "Brier": float(((probs - onehot) ** 2).sum(1).mean()),
            "accuracy": float((probs.argmax(1) == labels).mean())}


TEMPERATURE = fit_temperature(val_logits, y_val)
print(f"val’da topilgan temperature T = {TEMPERATURE:.3f}")
calib = pd.DataFrame({f"{split_name}, {label}": calibration_report(logits, labels, t)
                      for split_name, logits, labels in (("val", val_logits, y_val), ("test", test_logits, y_test))
                      for label, t in (("T=1", 1.0), (f"T={TEMPERATURE:.2f}", TEMPERATURE))}).T
display(calib.round(4))


def reliability_diagram(ax, probs, labels, title, n_bins=15):
    ece, bins = calibration_bins(probs, labels, n_bins)
    width = 1 / n_bins
    for lo, n, acc, conf in bins:
        ax.bar(lo, acc, width=width * 0.94, align="edge", color=CLASS_COLORS[0])
        ax.bar(lo, conf - acc, bottom=acc, width=width * 0.94, align="edge", color="none", edgecolor=CLASS_COLORS[1], hatch="///", lw=0)
        ax.text(lo + width / 2, 0.02, str(n), ha="center", fontsize=7, color="white" if acc > 0.08 else "#52514e", rotation=90)
    ax.plot([0, 1], [0, 1], "--", color="#52514e", lw=1)
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="ishonch (max softmax)", ylabel="haqiqiy aniqlik", title=f"{title}\nECE = {ece:.3f}")
    ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=CLASS_COLORS[0]),
                       plt.Rectangle((0, 0), 1, 1, facecolor="none", edgecolor=CLASS_COLORS[1], hatch="///")],
              labels=["aniqlik", "ishonch − aniqlik"], frameon=False, loc="upper left")


fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
reliability_diagram(axes[0], torch.softmax(test_logits, 1).numpy(), y_test, "Test, kalibrlashdan oldin (T = 1)")
reliability_diagram(axes[1], torch.softmax(test_logits / TEMPERATURE, 1).numpy(), y_test, f"Test, temperature scaling (T = {TEMPERATURE:.2f})")
plt.tight_layout()
plt.show()

## 8. Explainability: Grad-CAM va o‘sma maskasi

- **Swin-T → Grad-CAM** (Selvaraju va boshq., ICCV 2017): oxirgi bosqich (Stage 4, 7 × 7 × 768) chiqishiga hook qo‘yiladi. Bashorat qilingan sinf logit’ining shu aktivatsiyalar bo‘yicha gradienti fazoviy o‘rtachalanib kanal og‘irligini beradi; ReLU(Σ w·A) → 224 px ga kattalashtiriladi.
- **ViT-B/16 → attention rollout** (Abnar & Zuidema, ACL 2020): barcha qatlamlarning attention matritsalari (+ residual) ko‘paytiriladi, [CLS] → patch qatori 14 × 14 xarita beradi. U sinfga bog‘liq emas.

**Sanity check — dataset’dagi qo‘lda chizilgan o‘sma maskasi bilan:**
(a) issiqlik “energiyasi”ning maska ichidagi ulushi va maskaning maydon ulushi (tasodifiy xaritada ular teng bo‘ladi);
(b) pointing game (Zhang va boshq., ECCV 2016): xaritaning eng issiq nuqtasi maskaga (16 px tolerance bilan) tushadimi; tasodifiy nuqta uchun kutilgan ulush ham chiqariladi.

Ogohlantirish: 7 × 7 xarita qo‘pol (bitta katak 32 × 32 px). Grad-CAM — segmentatsiya emas va model “to‘g‘ri sababga ko‘ra to‘g‘ri” ekanining isboti ham emas; bu faqat tekshiruv vositasi.

In [ ]:
class SwinGradCAM:
    """Grad-CAM oxirgi Swin bosqichi uchun (chiqish NHWC: B × 7 × 7 × 768)."""

    def __init__(self, model):
        self.model = model
        self.handle = model.features[-1].register_forward_hook(self._hook)

    def _hook(self, module, inputs, output):
        self.activations = output
        output.register_hook(lambda grad: setattr(self, "gradients", grad))

    def __call__(self, x):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)
        logits.gather(1, logits.argmax(1, keepdim=True)).sum().backward()
        a, g = self.activations.permute(0, 3, 1, 2), self.gradients.permute(0, 3, 1, 2)   # → (B, C, h, w)
        cam = F.relu((g.mean(dim=(2, 3), keepdim=True) * a).sum(1, keepdim=True))
        return cam.detach(), logits.detach()

    def remove(self):
        self.handle.remove()


class ViTAttentionRollout:
    """Attention rollout: har bir qatlamning attention matritsasi (+ I, qatorlar normallashtirilgan) ko‘paytiriladi."""

    def __init__(self, model):
        self.model, self.maps, self.patched = model, [], []
        for layer in model.encoder.layers:
            mha = layer.self_attention
            original = mha.forward

            def forward(query, key, value, _original=original, **kwargs):
                kwargs.update(need_weights=True, average_attn_weights=False)
                out, weights = _original(query, key, value, **kwargs)
                self.maps.append(weights.detach())
                return out, weights

            mha.forward = forward
            self.patched.append(mha)

    @torch.no_grad()
    def __call__(self, x):
        self.maps = []
        logits = self.model(x)
        rollout = None
        for attention in self.maps:                                  # (B, heads, 197, 197)
            a = attention.mean(1)
            a = a + torch.eye(a.shape[-1], device=a.device)
            a = a / a.sum(-1, keepdim=True)
            rollout = a if rollout is None else a @ rollout
        side = int((rollout.shape[-1] - 1) ** 0.5)
        return rollout[:, 0, 1:].reshape(-1, 1, side, side), logits

    def remove(self):
        for mha in self.patched:
            del mha.forward                                          # asl forward metodiga qaytadi


explainer = SwinGradCAM(model) if MODEL_NAME == "swin_t" else ViTAttentionRollout(model)
METHOD = "Grad-CAM (Stage 4)" if MODEL_NAME == "swin_t" else "attention rollout"


def explain(idx):
    x, _ = get_batch(idx, train=False)
    cam, logits = explainer(x)
    cam = F.interpolate(cam.float(), size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)[:, 0]
    low = cam.amin(dim=(1, 2), keepdim=True)
    cam = (cam - low) / (cam.amax(dim=(1, 2), keepdim=True) - low).clamp(min=1e-8)     # har bir xarita [0, 1]
    return cam.cpu(), logits.float().cpu()


# Test slice’larida maska bilan solishtirish
model.eval()
eval_idx = np.random.default_rng(SEED).permutation(idx_test)[:CAM_EVAL_MAX]
radius = 16
energy_in, area, hit, hit_base, correct = [], [], [], [], []
for i in range(0, len(eval_idx), 16):
    batch = eval_idx[i:i + 16]
    cam, logits = explain(torch.as_tensor(batch, device=DEVICE))
    m = torch.from_numpy(masks[batch]).float()
    m_tol = F.max_pool2d(m[:, None], 2 * radius + 1, stride=1, padding=radius)[:, 0]   # maska + 16 px
    peak = cam.flatten(1).argmax(1)
    energy_in += ((cam * m).sum((1, 2)) / cam.sum((1, 2)).clamp(min=1e-8)).tolist()
    area += m.mean((1, 2)).tolist()
    hit += m_tol.flatten(1).gather(1, peak[:, None])[:, 0].tolist()
    hit_base += m_tol.mean((1, 2)).tolist()
    correct += (logits.argmax(1).numpy() == y[batch]).tolist()

cam_stats = {
    "slice soni": len(eval_idx),
    "issiqlikning maska ichidagi ulushi": float(np.mean(energy_in)),
    "maskaning maydon ulushi (tasodifiy xarita)": float(np.mean(area)),
    f"pointing game (maska + {radius} px)": float(np.mean(hit)),
    "tasodifiy nuqta uchun kutilgan": float(np.mean(hit_base)),
}
print(METHOD)
for key, value in cam_stats.items():
    print(f"{key:>44}: {value:.3f}" if isinstance(value, float) else f"{key:>44}: {value}")

In [ ]:
show_idx = []
for c in range(K):
    show_idx += list(np.random.default_rng(SEED + c).permutation(idx_test[y_test == c])[:3])
show_idx = np.array(show_idx[:9])
cam, logits = explain(torch.as_tensor(show_idx, device=DEVICE))
probs = torch.softmax(logits / TEMPERATURE, 1).numpy()
cols = 3
rows_n = math.ceil(len(show_idx) / cols)
fig, axes = plt.subplots(rows_n, cols * 2, figsize=(cols * 2 * 2.6, rows_n * 2.9))
axes = np.atleast_2d(axes)
for k, i in enumerate(show_idx):
    r, c = divmod(k, cols)
    ax_img, ax_cam = axes[r, 2 * c], axes[r, 2 * c + 1]
    ax_img.imshow(images[i], cmap="gray")
    ax_img.contour(masks[i], levels=[0.5], colors=["#1baf7a"], linewidths=1.2)
    ax_img.set_title(f"haqiqiy: {CLASSES[y[i]]}", fontsize=9)
    ax_cam.imshow(images[i], cmap="gray")
    ax_cam.imshow(cam[k], cmap="inferno", alpha=0.45, vmin=0, vmax=1)
    ax_cam.contour(masks[i], levels=[0.5], colors=["#1baf7a"], linewidths=1.2)
    p = probs[k].argmax()
    ax_cam.set_title(f"bashorat: {CLASSES[p]} ({probs[k, p]:.2f})", fontsize=9, color="#0b0b0b" if p == y[i] else "#e34948")
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle(f"{METHOD}; yashil kontur — dataset’dagi o‘sma maskasi; ehtimol — temperature scaling’dan keyin")
plt.tight_layout()
plt.show()
explainer.remove()

## 9. Eksport va MedNexus’ga ulash

Natijalar `OUT_DIR/export/` ga yoziladi:
- `model.safetensors` — `state_dict` (F32 og‘irliklar + Swin’ning I64 `relative_position_index` buferlari), metadata’da arxitektura, sinflar va T. Tekshirilgan: MedNexus’ning `app.analyzers.brain_mri.tumor.read_safetensors` funksiyasi bu faylni o‘qiydi va u `torchvision.models.swin_t(num_classes=3)` ga to‘liq yuklanadi;
- `model.onnx` — opset 17, dinamik batch, chiqishi — xom logit’lar (T ga bo‘lish ilovada), onnxruntime bilan PyTorch natijasiga solishtiriladi;
- `config.json` — sinflar tartibi, preprocessing, temperature, split va shu ishga tushirishda hisoblangan test metrikalari.

In [ ]:
def ensure(package, module=None):
    """Colab’da yo‘q bo‘lishi mumkin bo‘lgan paketni o‘rnatadi (faqat eksport uchun)."""
    if importlib.util.find_spec(module or package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


EXPORT_DIR = OUT_DIR / "export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
model.eval()

# --- safetensors ---
ensure("safetensors")
from safetensors.torch import load_file, save_file

state = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()}
save_file(state, EXPORT_DIR / "model.safetensors", metadata={
    "arch": MODEL_NAME, "classes": json.dumps(CLASSES), "img_size": str(IMG_SIZE),
    "temperature": f"{TEMPERATURE:.6f}", "dataset": "figshare 10.6084/m9.figshare.1512427 (CC BY 4.0)"})
reloaded, _ = build_model(MODEL_NAME, K, pretrained=False)
reloaded.load_state_dict(load_file(EXPORT_DIR / "model.safetensors"))
reloaded.eval()
x_check, _ = get_batch(tidx_test[:4], train=False)
with torch.no_grad():
    reference = model.float()(x_check).cpu()
    diff = (reloaded.to(DEVICE)(x_check).cpu() - reference).abs().max().item()
print(f"model.safetensors: {(EXPORT_DIR / 'model.safetensors').stat().st_size / 1e6:.1f} MB, qayta yuklangan model farqi {diff:.2e}")

# --- ONNX ---
ensure("onnx")
cpu_model = copy.deepcopy(model).cpu().eval()
export_kwargs = {"dynamo": False} if "dynamo" in inspect.signature(torch.onnx.export).parameters else {}   # TorchScript exporter
with warnings.catch_warnings():                 # torch >= 2.9: TorchScript exporter "legacy" deb ogohlantiradi
    warnings.simplefilter("ignore", DeprecationWarning)
    warnings.simplefilter("ignore", torch.jit.TracerWarning)      # ViT: o‘lcham tekshiruvi 224 px uchun konstanta bo‘lib qoladi
    torch.onnx.export(cpu_model, (torch.zeros(1, 3, IMG_SIZE, IMG_SIZE),), str(EXPORT_DIR / "model.onnx"),
                      input_names=["input"], output_names=["logits"], opset_version=17,
                      dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}}, **export_kwargs)
ensure("onnxruntime")
import onnxruntime as ort

session = ort.InferenceSession(str(EXPORT_DIR / "model.onnx"), providers=["CPUExecutionProvider"])
onnx_logits = session.run(None, {"input": x_check.cpu().numpy()})[0]
print(f"model.onnx: {(EXPORT_DIR / 'model.onnx').stat().st_size / 1e6:.1f} MB, onnxruntime vs PyTorch farqi "
      f"{np.abs(onnx_logits - reference.numpy()).max():.2e}")

# --- config.json ---
export_config = {
    "arch": MODEL_NAME, "classes": CLASSES, "img_size": IMG_SIZE, "temperature": TEMPERATURE,
    "preprocessing": {"input": "2D T1-CE MRI slice, grayscale", "intensity": f"per-image percentile min-max {list(CLIP_PERCENTILES)} → [0, 1]",
                      "geometry": "pad to square (black) → bilinear resize, uint8 → /255", "channels": "grayscale repeated to 3",
                      "normalize": {"mean": IMAGENET_MEAN, "std": IMAGENET_STD}},
    "output": "logits; probabilities = softmax(logits / temperature)",
    "dataset": {"name": "figshare brain tumor dataset (Jun Cheng)", "doi": "10.6084/m9.figshare.1512427", "license": "CC BY 4.0"},
    "split": {"mode": SPLIT_MODE, "seed": SEED, "train/val/test slices": [len(idx_train), len(idx_val), len(idx_test)],
              "train/val/test patients": [len(sets[s]) for s in ("train", "val", "test")]},
    "training": {"epochs_run": len(history), "best_epoch": checkpoint["epoch"], "lr": LR, "batch_size": BATCH_SIZE,
                 "label_smoothing": LABEL_SMOOTHING, "weight_decay": WEIGHT_DECAY, "freeze_stages": FREEZE_STAGES,
                 "smoke_test": SMOKE_TEST},
    "test_metrics_this_run": results,
    "calibration_test": calib.loc[[i for i in calib.index if i.startswith("test")]].round(4).to_dict(orient="index"),
}
(EXPORT_DIR / "config.json").write_text(json.dumps(export_config, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
archive = shutil.make_archive(str(OUT_DIR / f"brain_mri_{MODEL_NAME}_export"), "zip", EXPORT_DIR)
print("eksport:", sorted(p.name for p in EXPORT_DIR.iterdir()), "→", archive)

### MedNexus’ga ulash

MedNexus’da har bir model — bitta `Analyzer` klassi va `backend/analyzers.yaml` dagi bitta yozuv (frontend o‘zgarmaydi). Bu model **3 sinfli** va “no tumor” sinfi yo‘q, shuning uchun u mavjud 4 sinfli ViT modelini almashtirmaydi: u o‘sma bor deb topilgan slice uchun *o‘sma turi bo‘yicha ikkinchi fikr* sifatida ishlashi mumkin. Taklif (hali qo‘shilmagan):

```yaml
  # Swin-T, figshare brain tumor dataset (Jun Cheng, CC BY 4.0) da fine-tune qilingan; 3 sinf,
  # "no tumor" sinfi yo'q. Ehtimollar = softmax(logits / temperature), T val'da topilgan.
  - id: brain_mri_tumor_type_swin
    class: app.analyzers.brain_mri.tumor_type:SwinTumorTypeClassifier
    applies_to: [{modality: mri, region: head, views: [Axial, Coronal, Sagittal]}]
    params:
      weights_url: <model.safetensors joylashtirilgan URL, masalan GitHub release>
      weights_file: brain-tumor-swin-t-figshare.safetensors
      temperature: <T, config.json dan>
      thresholds: {report: 0.50, moderate: 0.70, high: 0.90}
```

Analyzer ichida preprocessing notebookdagi bilan **bir xil** bo‘lishi shart (`StudyImage.unit()` → `preprocess_slice` → /255 → 3 kanal → ImageNet mean/std), aks holda model boshqa taqsimotdagi kirishni ko‘radi:

```python
# Qisqartirilgan eskiz: __init__ (weights_url, weights_file, temperature, thresholds) va analyze()/result()
# mavjud tumor.py dagi BrainTumorClassifier kabi yoziladi.
class SwinTumorTypeClassifier(Analyzer):
    label = "Brain tumor type (Swin-T, figshare, 3 classes)"
    provides_findings = True
    CLASSES = ["Meningioma", "Glioma", "Pituitary tumor"]          # config.json dagi tartib

    def load(self, device):
        model = torchvision.models.swin_t(num_classes=3)
        model.load_state_dict(read_safetensors(ensure_weights(self.weights_url, self.weights_file)))
        self.model, self.device = model.eval().to(device), device

    @torch.inference_mode()
    def predict(self, image: StudyImage) -> dict[str, float]:
        x = mednexus_preprocess(image.unit()).to(self.device)        # quyidagi katakdagi funksiya
        probs = torch.softmax(self.model(x)[0] / self.temperature, dim=0)
        return dict(zip(self.CLASSES, probs.tolist()))
```

Quyidagi katak shu yo‘lni tekshiradi: asl `.mat` slice → ilovadagidek [0, 1] ga keltirilgan rasm → `mednexus_preprocess` → PyTorch va ONNX natijalari notebookdagi baholash yo‘li bilan bir xilmi.

In [ ]:
_mean, _std = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1), torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)


def mednexus_preprocess(unit_image):
    """StudyImage.unit() (H×W, [0, 1]) → (1, 3, 224, 224) model kirishi. Notebookdagi bilan bir xil."""
    x = torch.from_numpy(preprocess_slice(unit_image)).float().div(255)[None, None]
    return (x.expand(-1, 3, -1, -1) - _mean) / _std


i = int(idx_test[0])
raw, *_ = read_mat(MAT_BY_ID[int(slice_ids[i])])
unit = (raw - raw.min()) / max(raw.max() - raw.min(), 1e-6)     # ilova rasmni [0, 1] shkalada beradi
x_app = mednexus_preprocess(unit)
x_nb, _ = get_batch(torch.as_tensor([i], device=DEVICE), train=False)
with torch.no_grad():
    logits_app = cpu_model(x_app)
    logits_nb = cpu_model(x_nb.cpu())
logits_onnx = session.run(None, {"input": x_app.numpy()})[0]
print(f"kirish farqi (max |Δ|): {(x_app - x_nb.cpu()).abs().max():.4f}  — percentile normalizatsiya shkalaga bog‘liq emas; "
      "faqat uint8 yaxlitlashdagi farq qolishi mumkin")
print(f"logit farqi: PyTorch {(logits_app - logits_nb).abs().max():.2e}, ONNX {np.abs(logits_onnx - logits_nb.numpy()).max():.2e}")
app_probs = torch.softmax(torch.as_tensor(logits_onnx)[0] / TEMPERATURE, 0)
print("ilova ko‘rsatadigan ehtimollar (ONNX, T bilan):", {c: round(float(p), 3) for c, p in zip(CLASSES, app_probs)},
      "| haqiqiy:", CLASSES[y[i]])
print(f"\nanalyzers.yaml uchun: temperature: {TEMPERATURE:.4f}")

### Yakuniy xulosa (shu ishga tushirishdagi raqamlar)

In [ ]:
print(f"Dataset      : figshare brain tumor dataset (Jun Cheng), CC BY 4.0 — {len(y)} slice, {len(np.unique(pids))} bemor"
      + ("  [SMOKE TEST — raqamlar ma’nosiz]" if SMOKE_TEST else ""))
print(f"Split        : {SPLIT_MODE}, bemor darajasida — train/val/test = {len(idx_train)}/{len(idx_val)}/{len(idx_test)} slice, "
      f"{len(sets['train'])}/{len(sets['val'])}/{len(sets['test'])} bemor")
print(f"Model        : {MODEL_NAME} (torchvision, {'ImageNet-1K pretrained' if PRETRAINED else 'noldan'}), {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M parametr, "
      f"{IMG_SIZE} px")
print(f"O‘qitish     : {len(history)} epoch (eng yaxshisi {checkpoint['epoch']}), AdamW lr {LR}, cosine + warmup, "
      f"label smoothing {LABEL_SMOOTHING}, FREEZE_STAGES={FREEZE_STAGES}, AMP={USE_AMP}")
print(f"Test         : accuracy {results['accuracy']:.3f} (95% CI {results['accuracy_ci95']}), macro-F1 {results['macro_f1']:.3f} "
      f"(95% CI {results['macro_f1_ci95']}), macro ROC-AUC {results['macro_roc_auc_ovr']:.3f}, "
      f"bemor darajasida {results['patient_level_accuracy']:.3f} ({results['test_patients']} bemor)")
t1, tt = calib.loc["test, T=1"], calib.loc[f"test, T={TEMPERATURE:.2f}"]
print(f"Kalibrlash   : T = {TEMPERATURE:.3f}; test ECE {t1['ECE']:.3f} → {tt['ECE']:.3f}, NLL {t1['NLL']:.3f} → {tt['NLL']:.3f}")
print(f"Explainability: {METHOD}; issiqlikning maskadagi ulushi {cam_stats['issiqlikning maska ichidagi ulushi']:.3f} "
      f"(tasodifiy {cam_stats['maskaning maydon ulushi (tasodifiy xarita)']:.3f}), pointing game "
      f"{cam_stats[f'pointing game (maska + {radius} px)']:.3f} (tasodifiy {cam_stats['tasodifiy nuqta uchun kutilgan']:.3f})")

## 10. Cheklovlar (halol)

- **“No tumor” sinfi yo‘q.** figshare’dagi har bir slice’da o‘sma bor (maskasi bilan). Model normal miyani hech qachon ko‘rmagan va har qanday kirishga uchta o‘sma turidan birini tanlaydi — u “normal” deya olmaydi. Shuning uchun MedNexus’da u faqat o‘sma borligi aniqlangandan keyin *o‘sma turi* haqida fikr bera oladi. Keyingi qadam: Kaggle 4 sinfli to‘plam (1-bo‘limdagi ogohlantirishlar bilan: bemor ID’lari yo‘q, ichida figshare rasmlari bor) yoki T1-CE normal slice manbai.
- **Bitta manba.** Ikki xitoy shifoxonasi, 2005–2010 yillar, bitta protokol. Boshqa skaner, protokol va populyatsiyada aniqlik pasayishi mumkin; bu notebookda tashqi (external) test yo‘q.
- **Faqat T1-CE.** T2, FLAIR, DWI va boshqa ketma-ketliklar model uchun taqsimotdan tashqari. (MedNexus’dagi mavjud modelda ham T2 glioblastoma meningioma deb baholangan — bu kutilgan xato turi.)
- **2D slice.** Model bitta slice’ni ko‘radi, 3D hajm konteksti yo‘q. Slice’larni dataset mualliflari tanlagan (o‘sma ko‘rinadigan kesimlar), shuning uchun real tekshiruvdagi ixtiyoriy slice’lar boshqacha taqsimotda.
- **Faqat 3 tur.** Metastaz, limfoma, abscess, insult va boshqa patologiyalar dataset’da yo‘q — model ularni ham shu uch sinfdan biriga kiritadi.
- **Kichik test.** ~35 bemor → ishonch oralig‘i keng; natija bitta split va bitta seed’ga bog‘liq. `SPLIT_MODE = "cvind"` bilan 5 fold va bir necha seed bo‘yicha takrorlash tavsiya etiladi.
- **Grad-CAM qo‘pol va isbot emas** (8-bo‘lim).
- Tadqiqot prototipi; sertifikatlangan tibbiy vosita emas.

**Keyingi qadamlar:** (1) “no tumor” sinfini qo‘shish; (2) `cvind` bo‘yicha to‘liq 5-fold CV va fold’lar ansambli; (3) boshqa markaz ma’lumotida external test; (4) BraTS bilan 3D, ko‘p ketma-ketlikli model.

## 11. Manbalar

1. Cheng, Jun (2017). *brain tumor dataset*. figshare. Dataset. <https://doi.org/10.6084/m9.figshare.1512427> (v8). Litsenziya: CC BY 4.0.
2. Cheng J, Huang W, Cao S, Yang R, Yang W, Yun Z, Wang Z, Feng Q. *Enhanced Performance of Brain Tumor Classification via Tumor Region Augmentation and Partition.* PLoS ONE 10(10): e0140381, 2015. — bemor darajasidagi 5-fold CV; accuracy 91,28 % gacha (BoW + kengaytirilgan o‘sma ROI + ring-form partition; test vaqtida qo‘lda chizilgan o‘sma chegarasi kerak).
3. Cheng J, va boshq. *Retrieval of Brain Tumors by Adaptive Spatial Pooling and Fisher Vector Representation.* PLoS ONE 11(6), 2016.
4. Badža MM, Barjaktarović MČ. *Classification of Brain Tumors from MRI Images Using a Convolutional Neural Network.* Applied Sciences 10(6): 1999, 2020. — shu datasetda record-wise va subject-wise 10-fold CV; eng yaxshi natija 96,56 % record-wise + augmentation holatida.
5. Liu Z, Lin Y, Cao Y, Hu H, Wei Y, Zhang Z, Lin S, Guo B. *Swin Transformer: Hierarchical Vision Transformer using Shifted Windows.* ICCV 2021.
6. Dosovitskiy A, va boshq. *An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale.* ICLR 2021.
7. Selvaraju RR, Cogswell M, Das A, Vedantam R, Parikh D, Batra D. *Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization.* ICCV 2017.
8. Abnar S, Zuidema W. *Quantifying Attention Flow in Transformers.* ACL 2020.
9. Zhang J, Lin Z, Brandt J, Shen X, Sclaroff S. *Top-down Neural Attention by Excitation Backprop.* ECCV 2016 (pointing game).
10. Guo C, Pleiss G, Sun Y, Weinberger KQ. *On Calibration of Modern Neural Networks.* ICML 2017.
11. Müller R, Kornblith S, Hinton G. *When Does Label Smoothing Help?* NeurIPS 2019.
12. Loshchilov I, Hutter F. *Decoupled Weight Decay Regularization* (AdamW). ICLR 2019; *SGDR: Stochastic Gradient Descent with Warm Restarts* (cosine schedule). ICLR 2017.
13. Nickparvar M. *Brain Tumor MRI Dataset.* Kaggle, v2 (CC BY 4.0): <https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset>
14. Menze BH, va boshq. *The Multimodal Brain Tumor Image Segmentation Benchmark (BRATS).* IEEE Transactions on Medical Imaging 34(10), 2015.
15. torchvision modellari va og‘irliklari: <https://pytorch.org/vision/stable/models.html>